<a href="https://colab.research.google.com/github/AwadZafar5493/Pytorch-Tutorial/blob/main/Paper01_CNN_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Paper Title: Deep learning algorithms for intrusion detection systems in internet of things using CIC-IDS 2017 dataset

In [2]:
# Steps: 01- Data Loading, 02- Prepocessing, 03- CNN Model, 04- LSTM Model, 05- Training

In [3]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
# Mount Google Drive
#from google.colab import drive
#drive.mount('/content/drive')

In [5]:
# Combine all files
#import os
#import pandas as pd

#folder_path = "/content/drive/MyDrive/CIC-IDS2017/MachineLearningCVE"

#all_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

#print("Files found:", all_files)

#df_list = []

#for file in all_files:
 #   file_path = os.path.join(folder_path, file)
  #  print("Loading:", file_path)

   # df = pd.read_csv(file_path)
    #df_list.append(df)

#df = pd.concat(df_list, ignore_index=True)

#print("Final dataset shape:", df.shape)

In [6]:
df.head(20)

NameError: name 'df' is not defined

In [ ]:
df.info()

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df['Label'].value_counts()

In [ ]:
# Save Combined File to Google Drive
#save_path = "/content/drive/MyDrive/CIC-IDS2017/combined_dataset.csv"

#df.to_csv(save_path, index=False)

#print("Saved to:", save_path)

In [ ]:
# ==============================================
# Step 2: Load Combined Dataset
# ==============================================
file_path = "/content/drive/MyDrive/CIC-IDS2017/combined_dataset.csv"
df = pd.read_csv(file_path)

# Strip spaces from column names
df.columns = df.columns.str.strip()

# Drop NaN / Infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print("Dataset shape after cleaning:", df.shape)
print(df['Label'].value_counts())

In [ ]:
# ==============================================
# Step 3: Preprocessing Features and Labels
# ==============================================

# Separate features and labels
X = df.drop("Label", axis=1).values
y = df['Label'].values

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y)  # Converts attack names to integers

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to torch tensors
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create DataLoader
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Number of classes:", len(le.classes_))
print("Number of features:", X.shape[1])

In [ ]:
# ==============================================
# Step 4a: Define CNN Model
# ==============================================
class TabularCNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(TabularCNN, self).__init__()
        self.input_dim = input_dim

        # Reshape input to [batch, 1, sqrt_dim, sqrt_dim]
        self.sqrt_dim = int(np.ceil(np.sqrt(input_dim)))
        self.pad = self.sqrt_dim**2 - input_dim  # padding for non-square

        # Conv layers
        self.conv1 = nn.Conv2d(1, 32, 3, 1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * (self.sqrt_dim//2) * (self.sqrt_dim//2), 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Pad and reshape
        if self.pad > 0:
            x = F.pad(x, (0, self.pad), "constant", 0)
        x = x.view(-1, 1, self.sqrt_dim, self.sqrt_dim)

        # Conv layers
        x = F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))

        # Flatten
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [ ]:
# ==============================================
# Step 4b: Define LSTM Model
# ==============================================
class TabularLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, num_classes):
        super(TabularLSTM, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # LSTM layer
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

        # Fully connected output
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # Reshape input to [batch, seq_len, features]
        x = x.unsqueeze(1)  # seq_len = 1
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim).to(device)

        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]  # take last output
        out = self.fc(out)
        return F.log_softmax(out, dim=1)

In [ ]:
# Step 05- TRAIN Function

def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses, test_losses = [], []

    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            predicted = outputs.argmax(dim=1)
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

        train_acc = correct/total*100
        train_losses.append(running_loss/len(train_loader))

        # Evaluate
        model.eval()
        correct = 0
        total = 0
        eval_loss = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                eval_loss += loss.item()
                predicted = outputs.argmax(dim=1)
                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)
        test_acc = correct/total*100
        test_losses.append(eval_loss/len(test_loader))

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Test Loss: {eval_loss/len(test_loader):.4f} | Test Acc: {test_acc:.2f}%")

    return train_losses, test_losses

In [ ]:
# Step 06- Instantiate and Train CNN
num_classes = len(le.classes_)
input_dim = X.shape[1]

cnn_model = TabularCNN(input_dim=input_dim, num_classes=num_classes)
train_model(cnn_model, train_loader, test_loader, epochs=5, lr=0.001)